In [1]:
from pandas import DataFrame
import torch
import numpy as np
from dataLoader import *
from MultiOutputTuner import *
from pathlib import Path
from BaselineRegressor import BaselineRegressor
from sklearn.metrics import mean_squared_error
from MultiOutputTuner import MultiOutputTuner
from sklearn.preprocessing import StandardScaler

In [13]:
X_train = load_msi_data(HSI_SATELLITE_DIR)
X_train_ebm = load_msi_data(MSI_SATELLITE_TRAIN_DIR)
y_train = gt_train_df[column_names].values

baseline = BaselineRegressor()
baseline.fit(X_train, y_train)

y_baseline = baseline.predict(X_train)
mse_baseline = mean_squared_error(y_train, y_baseline)
mse_baseline

692.7846086554895

In [3]:
classic_models_dir = Path("./models/hsi")
classic_results = {}

for model_path in classic_models_dir.iterdir():
    if(model_path.is_dir()):
        continue
    print(model_path) 
    model = MultiOutputTuner.load(model_path)
    if model_path.name.startswith("ebm"):
        y_pred = model.predict(X_train_ebm)
        
    elif model_path.is_file():
        y_pred = model.predict(X_train)

    mse = mean_squared_error(y_train,y_pred, multioutput='raw_values')
    classic_results[model_path.name] = mse/mse_baseline
df_classic_results = pd.DataFrame(classic_results).T
df_classic_results.index.name = "model"
print(column_names)
df_classic_results

models/hsi/gradientBoosting
models/hsi/kNeighbors
models/hsi/adaBoost
models/hsi/gradientBoosting_95
models/hsi/xgb
models/hsi/xgb_95
models/hsi/kNeighbors_95
models/hsi/adaBoost_95
models/hsi/xgb_975
models/hsi/xgb_9
models/hsi/ebm
models/hsi/ebm_95
models/hsi/randomForest
models/hsi/randomForest_95
['Fe', 'Zn', 'B', 'Cu', 'S', 'Mn']


,0,1,2,3,4,5
model,,,,,,
gradientBoosting,0.404159,0.317792,0.368343,0.479734,0.462124,0.464421
kNeighbors,0.442626,0.384154,0.402718,0.441975,0.401692,0.395866
adaBoost,0.721241,0.766127,0.712064,0.837783,0.760669,0.700646
gradientBoosting_95,0.718645,0.874904,0.758972,0.859713,0.733984,0.759633
xgb,0.416040,0.308843,0.577134,0.528808,0.406467,0.329684
xgb_95,0.744248,0.762981,0.756419,0.853232,0.785052,0.755086
kNeighbors_95,0.754000,0.826905,0.769930,0.857200,0.811943,0.427419
adaBoost_95,0.754134,0.859632,0.779466,0.876514,0.800472,0.781339
xgb_975,0.715136,0.828904,0.759398,0.849009,0.784504,0.694284


CNN2D 0.95 =     0.53048769, 0.66062731, 0.50172776, 0.72228935, 0.65769953, 0.58475807
CNN3D 0.95 =     0.49918575, 0.62049721, 0.45977483, 0.68627737, 0.61806255, 0.53619069
CNN2D =          0.46976451, 0.5716636 , 0.42982737, 0.66206401, 0.55735863, 0.49101401
RNN  0.95=       0.4231152 , 0.41313581, 0.37148547, 0.51408543, 0.5350001, 0.55188089
RNN =            0.42545757, 0.47854947, 0.38916031, 0.59854028, 0.56769386, 0.52527768
BandRNN =        0.9998284 , 1.00009714, 1.00006329, 1.0000605 , 0.99993886, 0.99997799
BandRNN 0.95 =   0.49994122, 0.67954018, 0.52331478, 0.74543027, 0.66064053, 0.57780528
FusionRNN 0.95 = 0.3984208 , 0.41592714, 0.30646235, 0.49188126, 0.50427348, 0.4238029
FusionRNN =      0.39059347, 0.39539525, 0.29104869, 0.45612823, 0.49257277, 0.40915999

# Final training model

In [4]:
from neural_networks.Recurrent import FusionNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FusionNN(neurons=256, hidden_size=12000).to(device)
model = torch.load_state_dict(torch.load("models/nn/FusionNN.pt"))

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


AttributeError: module 'torch' has no attribute 'load_state_dict'

# Final testing project

In [3]:
import torch
import torch.nn as nn
from neural_networks.Recurrent import FusionNN
from neural_networks.Convolutional import CNN2D

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fusion_model = FusionNN(neurons=256, hidden_size=12000).to(device)
fusion_model.load_state_dict(torch.load("models/nn/FusionNN.pt"))

convolutional_model = CNN2D(neurons=256, channels=231).to(device)
convolutional_model.load_state_dict(torch.load("models/nn/CNN2D_HSI.pt"))

class CombinedModel(nn.Module):
    def __init__(self, cnn_hsi, fusion_model):
        super().__init__()
        self.cnn_hsi = cnn_hsi
        self.fusion_model = fusion_model

    def forward(self, msi_x, hsi_x):
        # 1. Obsługa przypadków gdy dane przychodzą jako lista/tuple z DataLoader lub load_images
        if isinstance(msi_x, (list, tuple)):
            msi_x = torch.stack(msi_x)
        if isinstance(hsi_x, (list, tuple)):
            hsi_x = torch.stack(hsi_x)

        # 2. Obliczenie wyniku z FusionNN (oczekuje listy [msi, hsi])
        out_fusion = self.fusion_model([msi_x, hsi_x]) # Kształt: [B, 6]

        # 3. Obliczenie wyniku z osobnego modelu CNN2D dla HSI
        out_cnn = self.cnn_hsi(hsi_x)                  # Kształt: [B, 6]

        # 4. Poprawne połączenie z zachowaniem kolejności:
        # Index 0, 1, 3 -> z CNN
        # Index 2, 4, 5 -> z Fusion
        combined_output = torch.empty_like(out_cnn)
        
        combined_output[:, 0] = out_cnn[:, 0]
        combined_output[:, 1] = out_cnn[:, 1]
        combined_output[:, 2] = out_fusion[:, 2]
        combined_output[:, 3] = out_cnn[:, 3]
        combined_output[:, 4] = out_fusion[:, 4]
        combined_output[:, 5] = out_fusion[:, 5]

        return combined_output
# CNN2d 0.5292173004349948	0.3796 0.6784 0.4795 0.4752 0.6130 0.5496 Cu Zn Fe
# Fusion 0.5456544568633203    0.3603 0.7045 0.5321 0.5361 0.6049 0.5361 B S Mn

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


In [4]:
from torch.utils.data import Dataset

class MultiSourceDataset(Dataset):
    def __init__(self, msi, hsi, airborne, targets):
        self.msi = msi
        self.hsi = hsi
        self.airborne = airborne
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return (
            self.msi[idx],
            self.hsi[idx],
            self.airborne[idx],
            self.targets[idx]
        )

In [5]:
from torch.utils.data import DataLoader, TensorDataset

msi_train= load_images(MSI_SATELLITE_TRAIN_DIR)
hsi_train= load_images(HSI_SATELLITE_DIR)

msi_train = torch.stack(msi_train)
hsi_train = torch.stack(hsi_train)

scaler = StandardScaler()
y = scaler.fit_transform(y_train)

fusion_model = FusionNN(neurons=256, hidden_size=12000).to(device)
fusion_model.load_state_dict(torch.load("models/nn/FusionNN.pt"))

convolutional_model = CNN2D(neurons=256, channels=231).to(device)
convolutional_model.load_state_dict(torch.load("models/nn/CNN2D_HSI.pt"))

final_model = CombinedModel(convolutional_model, fusion_model).to(device)

# 1. Konwersja y z NumPy array na PyTorch Tensor
y_tensor = torch.tensor(y, dtype=torch.float32)

# 2. Utworzenie TensorDataset z użyciem y_tensor
dataset = TensorDataset(msi_train, hsi_train, y_tensor)

# 3. Definicja DataLoader
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [6]:
import torch.optim as optim

final_model.train()

# 2. Definicja funkcji straty i optymalizatora (optymalizuje parametry obu sieci)
criterion = nn.MSELoss()
optimizer = optim.Adam(final_model.parameters(), lr=1e-5)

best_loss = float("inf")
early_stopping_counter = 0
early_stopping_treshold = 16 
# 3. Pętla treningowa (przykład dla DataLoader)
for epoch in range(1000):
    for msi_batch, hsi_batch, targets in dataloader:
        msi_batch = msi_batch.to(device)
        hsi_batch = hsi_batch.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        
        # Wykonanie predykcji
        predictions = final_model(msi_batch, hsi_batch)
        
        # Obliczenie straty dla wszystkich 6 wyjść jednocześnie
        loss = criterion(predictions, targets)
        val_loss = loss.item()
        
        # Wsteczna propagacja i krok optymalizatora
        loss.backward()
        optimizer.step()

    print(epoch, val_loss)
    if val_loss < best_loss:
        best_loss = val_loss
        early_stopping_counter = 0
        if val_loss < 0.24:
            break

    else:
        early_stopping_counter += 1
        
        if early_stopping_counter >=early_stopping_treshold:
            break

0 0.5344672203063965
1 0.4896758198738098
2 0.23479056358337402


In [9]:
scaler = StandardScaler()
y = scaler.fit_transform(y_train)

# fusion_model = FusionNN(neurons=256, hidden_size=12000).to(device)
# fusion_model.load_state_dict(torch.load("models/nn/FusionNN.pt"))

# convolutional_model = CNN2D(neurons=256, channels=231).to(device)
# convolutional_model.load_state_dict(torch.load("models/nn/CNN2D_HSI.pt"))

# final_model = CombinedModel(convolutional_model, fusion_model).to(device)
final_model.eval()

msi_imgs = load_images(MSI_SATELLITE_TEST_DIR)
hsi_imgs = load_images(HSI_SATELLITE_TEST_DIR) 

msi_batch = torch.stack(msi_imgs).to(device)
hsi_batch = torch.stack(hsi_imgs).to(device)

# 4. Predykcja
with torch.no_grad():
    predictions = final_model(msi_batch, hsi_batch).cpu().numpy()
    predictions = scaler.inverse_transform(predictions)

df = DataFrame(data = predictions, columns=column_names)
df.to_csv("final_trained.csv", index_label="sample_index")

In [14]:
msi_imgs = load_images(MSI_SATELLITE_TRAIN_DIR)
hsi_imgs = load_images(HSI_SATELLITE_DIR) 

msi_batch = torch.stack(msi_imgs).to(device)
hsi_batch = torch.stack(hsi_imgs).to(device)

# 4. Predykcja
with torch.no_grad():
    predictions = final_model(msi_batch, hsi_batch).cpu().numpy()
    predictions = scaler.inverse_transform(predictions)
mse = mean_squared_error(y_train, predictions)
score = mse / mse_baseline
score

0.4547759066955624